<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 3 (b) — Build a Document Q&A

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Build

Pick a document. Ask it questions. Get answers **grounded in the document**, with sources.

```
your document  →  chunk  →  embed  →  store        (once)
your question  →  embed  →  search  →  prompt  →  answer   (every time)
```

1. Choose a document and chunk it properly
2. Index the chunks in Chroma
3. Write `search(question)` and check it returns sensible chunks
4. Write `answer(question)` — retrieve, then let the model read
5. *(stretch)* wrap it in a Gradio chat, and make it say "I don't know"

> **You need an OpenAI key** for step 4 onwards. Steps 1–3 run without one.

---

## 1. Setup

Run these three cells. Nothing to write yet.

In [ ]:
# PROVIDED - just run this cell.
!pip install -q chromadb sentence-transformers litellm langchain-text-splitters gradio

In [ ]:
# PROVIDED - just run this cell.
import os
from getpass import getpass
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from litellm import completion
import chromadb
import gradio as gr

model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
print("Ready")

In [ ]:
# PROVIDED - only needed from step 4 onwards. Press Enter to skip for now.
key = getpass("OpenAI API Key (Enter to skip): ")
if key:
    os.environ["OPENAI_API_KEY"] = key
    print("Key loaded")
else:
    print("No key - steps 1 to 3 will still work")

---

## 2. Build it

**The tools you have:**

| What | How |
|---|---|
| split text | `RecursiveCharacterTextSplitter(chunk_size=…, chunk_overlap=…).split_text(text)` |
| make vectors | `model.encode(list_of_texts).tolist()` — the `.tolist()` matters, Chroma won't take numpy |
| store | `collection.add(ids=…, embeddings=…, documents=…, metadatas=…)` |
| search | `collection.query(query_embeddings=…, n_results=…)` → `["documents"][0]` |
| ask the model | `completion(model="gpt-4o-mini", messages=[…], temperature=0)` |

Everything from here is yours to write.

In [ ]:
# Step 1 - Put your document in a string called DOCUMENT.
# Two or three paragraphs minimum. A Wikipedia article, a policy page, your own notes.
# Then chunk it and print how many chunks you got, plus one of them.

In [ ]:
# Step 2 - Index the chunks in a Chroma collection.
# Give each chunk an id. Print collection.count() to prove it worked.

In [ ]:
# Step 3 - Write search(question, k=3) that returns the k most relevant chunk texts.
# Test it with a question whose words do NOT appear in your document.

In [ ]:
# Step 4 - Write answer(question): retrieve the chunks, put them in the system prompt,
# and return the model's reply. Tell the model to use ONLY that context.
# Use temperature=0 - you want it reading, not inventing.

In [ ]:
# Step 5 - Ask it three questions you know the answer to, and one it CANNOT know.
# What does it do with the last one?

In [ ]:
# Step 6 (stretch) - Wrap it in a gr.ChatInterface so you can talk to your document.
# Hint: def chat(message, history): return answer(message)

In [ ]:
# Step 7 (stretch) - Make it honest. Look at the `distances` Chroma returns and,
# if the closest chunk is further away than some cut-off you pick,
# return "I don't know" WITHOUT calling the model at all.
# How did you choose the number? That question is tomorrow's lesson.

---

**When it misbehaves:**

| What you see | What it means |
|---|---|
| `ValueError` about embeddings on `add()` | you passed a numpy array — add `.tolist()` |
| `IDs already exist` | you ran `add()` twice; use a new collection name or restart the runtime |
| `results["documents"]` looks doubly nested | it is — `query` takes a *batch*, so index `[0]` |
| The answer ignores your document | your context probably isn't in the **system** message, or `search` returned nothing |
| It confidently answers the impossible question | expected! Similarity search always returns *something*. That's step 7 — and Day 4 |
| `AuthenticationError` | re-run the key cell |

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **Chunk, embed, store** | the indexing half — done once, offline |
| **Embed, search, prompt** | the querying half — done on every question |
| **Same model both sides** | documents and queries must share one coordinate space |
| **`temperature=0`** | for RAG the model reports, it doesn't create |
| **Grounding** | "use ONLY this context" is the difference between reading and guessing |
| **Top-k always returns k** | search cannot tell you that nothing is relevant — you have to decide that yourself |

**Finished early?**
1. Add `metadatas` with a section name, and print the source under each answer.
2. Index the same document at two chunk sizes and compare answers to the same five questions.
3. Add `share=True` to `launch()` and let someone else question your document.